# Triton Kernel 主线 · 第 10/10 课：无状态随机数与 Dropout

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现可复现 inverted dropout，解释 seed+offset 的确定性和并行安全。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：Triton 常用 counter-based RNG：随机值由 seed 与元素 offset 决定，不维护跨线程共享状态。

## 核心心智模型

### 1. 它是什么，解决什么问题

Triton 常用 counter-based RNG：随机值由 seed 与元素 offset 决定，不维护跨线程共享状态。

### 2. 它如何工作

keep=rand>p；训练时保留值除以 1-p，使输出期望等于输入；同 seed/offset 重算同一 mask。

### 3. 正确性条件与常见误区

必须约束 0≤p<1；不同算子/层要避免复用相同 seed-offset 域导致相关 mask。

### 4. 性能与工程取舍

不物化 mask 节省内存，但 backward 必须用相同映射重算；改变 grid 映射可能改变随机序列。

## 具体演示

p=.2 时保留值乘 1.25；全 1 输入的大样本均值应接近 1。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 keep mask。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def dropout_kernel(x, out, n: tl.constexpr, p, seed,
                   BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < n
    xv = tl.load(x + offsets, mask=mask)
    keep = ______  # TODO: 每元素无状态 Bernoulli
    y = tl.where(keep, xv / (1.0 - p), 0.0)
    tl.store(out + offsets, y, mask=mask)

def dropout(x, p, seed):
    assert x.is_contiguous() and 0.0 <= p < 1.0
    out = torch.empty_like(x)
    n = x.numel()
    dropout_kernel[(triton.cdiv(n, 256),)](x, out, n, p, seed, BLOCK=256)
    return out

x = torch.ones(10003, device="cuda")
y1, y2 = dropout(x, .2, 123), dropout(x, .2, 123)
assert torch.equal(y1, y2)
assert set(torch.unique(y1).tolist()).issubset({0.0, 1.25})
assert abs(y1.mean().item() - 1.0) < .05


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“无状态随机数与 Dropout”的工作机制。

**你的答案：**


### Q2

为什么相同 seed 不同 offsets 能并行无锁地生成随机数？

**你的答案：**


### Q3

融合后 offset 映射改变，如何保持 forward/backward mask 一致？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。